## Imports

In [11]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.ndimage.filters as filters

## Load in images

In [ ]:
folder_path = Path("./materialy_feature_points")
extensions = ["*.png", "*.jpg"]

image_paths = []
for ext in extensions:
    image_paths.extend(folder_path.glob(ext))

images = []
for path in image_paths:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    
    images.append(img)

## Harris Method

In [21]:
fountain_1 = images[4]
fountain_2 = images[5]

In [22]:
def find_max(image, size, threshold) : # size - maximum filter mask size
    data_max = filters.maximum_filter(image, size)
    maxima = (image == data_max)
    diff = image > threshold
    maxima[diff == 0] = 0
    return np.nonzero(maxima)

In [23]:
def get_h_from_auto_corelation_matrix(image, sobel_size, gauss_size, K=0.05):
    # Gradient with Sobel
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=sobel_size)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=sobel_size)

    I2x = Ix**2
    I2y = Iy**2
    Ixy = Ix * Iy

    # Applying Gaussian blur
    blurred_I2x = cv2.GaussianBlur(I2x, gauss_size, sigmaX=1)
    blurred_I2y = cv2.GaussianBlur(I2y, gauss_size, sigmaX=1)
    blurred_Ixy = cv2.GaussianBlur(Ixy, gauss_size, sigmaX=1)

    auto_corelation_matrix = np.array([
        [blurred_I2x, blurred_Ixy],
        [blurred_Ixy, blurred_I2y]
    ])

    H = np.det(auto_corelation_matrix) - K * (np.trace(auto_corelation_matrix)) ** 2

    return find_max(H, max(sobel_size, gauss_size), 127)

In [24]:
get_h_from_auto_corelation_matrix(fountain_1, 3, 5)

error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'GaussianBlur'
> Overload resolution failed:
>  - Can't parse 'ksize'. Input argument doesn't provide sequence protocol
>  - Can't parse 'ksize'. Input argument doesn't provide sequence protocol
